# 00 Exploração das fontes

Conhecer a estrutura das vendas e dos canais e identificar condições que precisam de tratamento.

### Leitura das fontes

Disponibiliza os dois arquivos para análise e confere se possuem as colunas esperadas.

In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/workspace/beverage_sales_landing/source_files"
SALES_PATH = f"{LANDING}/abi_bus_case1_beverage_sales_20210726.csv"
CHANNELS_PATH = f"{LANDING}/abi_bus_case1_beverage_channel_group_20210726.csv"
SALES_COLUMNS = [
    "DATE", "CE_BRAND_FLVR", "BRAND_NM", "Btlr_Org_LVL_C_Desc",
    "CHNL_GROUP", "TRADE_CHNL_DESC", "PKG_CAT", "Pkg_Cat_Desc",
    "TSR_PCKG_NM", "$ Volume", "YEAR", "MONTH", "PERIOD",
]
CHANNEL_COLUMNS = ["TRADE_CHNL_DESC", "TRADE_GROUP_DESC", "TRADE_TYPE_DESC"]


def read_source(path, separator, expected_columns, encoding, multiline=False):
    frame = (
        spark.read.option("header", True)
        .option("sep", separator)
        .option("inferSchema", False)
        .option("encoding", encoding)
        .option("multiLine", multiline)
        .option("ignoreLeadingWhiteSpace", False)
        .option("ignoreTrailingWhiteSpace", False)
        .option("mode", "FAILFAST")
        .csv(path)
    )
    if frame.columns != expected_columns:
        raise ValueError(f"Cabeçalho inesperado em {path}: {frame.columns}")
    return frame


sales_raw = read_source(SALES_PATH, "\t", SALES_COLUMNS, "UTF-16", multiline=True)
channels_raw = read_source(CHANNELS_PATH, ",", CHANNEL_COLUMNS, "UTF-8")
print("Leitura sem inferência de tipos: valores mantidos como texto.")
print("FAILFAST interrompe a leitura se o parser encontrar registros malformados.")

Leitura sem inferência de tipos: valores mantidos como texto.
FAILFAST interrompe a leitura se o parser encontrar registros malformados.


## 1 Estrutura e amostras

Confere a quantidade de registros e colunas de cada fonte. As amostras permitem reconhecer os campos e o conteúdo recebido.

In [0]:
sales_count = sales_raw.count()
channels_count = channels_raw.count()
display(spark.createDataFrame(
    [
        ("sales", sales_count, len(sales_raw.columns), 16151, sales_count == 16151),
        ("channel_group", channels_count, len(channels_raw.columns), 31, channels_count == 31),
    ],
    "fonte string, registros long, colunas int, referencia_zip long, contagem_confere boolean",
))
sales_raw.printSchema()
channels_raw.printSchema()
display(sales_raw.limit(20))
display(channels_raw)

fonte,registros,colunas,referencia_zip,contagem_confere
sales,16151,13,16151,true
channel_group,31,3,31,true


root
 |-- DATE: string (nullable = true)
 |-- CE_BRAND_FLVR: string (nullable = true)
 |-- BRAND_NM: string (nullable = true)
 |-- Btlr_Org_LVL_C_Desc: string (nullable = true)
 |-- CHNL_GROUP: string (nullable = true)
 |-- TRADE_CHNL_DESC: string (nullable = true)
 |-- PKG_CAT: string (nullable = true)
 |-- Pkg_Cat_Desc: string (nullable = true)
 |-- TSR_PCKG_NM: string (nullable = true)
 |-- $ Volume: string (nullable = true)
 |-- YEAR: string (nullable = true)
 |-- MONTH: string (nullable = true)
 |-- PERIOD: string (nullable = true)

root
 |-- TRADE_CHNL_DESC: string (nullable = true)
 |-- TRADE_GROUP_DESC: string (nullable = true)
 |-- TRADE_TYPE_DESC: string (nullable = true)



DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,$ Volume,YEAR,MONTH,PERIOD
1/1/2006,3440,LEMON,CANADA,LEISURE,SPORT VENUE,N20O,20Z/600ML,.591L NRP 24L,22.48,2006,1,1
1/1/2006,3440,LEMON,NORTHEAST,SUPERS,SUPERETTE,N20O,20Z/600ML,20Z NRP 24L,100,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHEAST,WORKPLACE,PLANT / OFFICE,N20O,20Z/600ML,20Z NRP 24L,66.14,2006,1,1
1/1/2006,3441,RASPBERRY,MIDWEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,222.5,2006,1,1
1/1/2006,3440,LEMON,WEST,MASS MERCHANDISER,MASS MERCHANDISER,N20O,20Z/600ML,20Z NRP 24L,302.5,2006,1,1
1/1/2006,3440,LEMON,MIDWEST,OTHER SMALL STORES,LIQUOR/BEER/WINE/SOFT,N20O,20Z/600ML,20Z NRP 24L,10,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1
1/1/2006,3554,STRAWBERRY,CANADA,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,.591L NRP 24L,59.95,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHWEST,SUPERS,SUPERMARKET,N20O,20Z/600ML,20z NRP 24L S,300,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHEAST,FOOD SERVICE,QUICK SERVICE RESTAURANT,N20O,20Z/600ML,20Z NRP 24L,85,2006,1,1


TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC
SPORT VENUE,ENTERTAINMENT,ALCOHOLIC
SUPERETTE,SERVICES,MIX
PLANT / OFFICE,SERVICES,MIX
MASS MERCHANDISER,GROCERY,MIX
LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC
CONVENIENCE STORE,SERVICES,MIX
QUICK SERVICE RESTAURANT,ENTERTAINMENT,ALCOHOLIC
SUPERMARKET,GROCERY,MIX
ALL OTHER,OTHER,MIX
OTHER EATING + DRINKING,SERVICES,MIX


## 2 Completude e diversidade dos valores

Identifica campos nulos, vazios e com espaços nas bordas. A quantidade de valores distintos ajuda a reconhecer códigos e categorias.

In [0]:
def column_profile(frame, source_name):
    expressions = []
    for index, name in enumerate(frame.columns):
        value = F.col(name)
        expressions.extend([
            F.count(F.when(value.isNull(), 1)).alias(f"null_{index}"),
            F.count(F.when(value.isNotNull() & (F.trim(value) == ""), 1)).alias(f"blank_{index}"),
            F.count(F.when(value != F.trim(value), 1)).alias(f"space_{index}"),
            F.countDistinct(value).alias(f"distinct_{index}"),
            F.countDistinct(F.trim(value)).alias(f"trim_distinct_{index}"),
        ])
    # Somente uma linha agregada é trazida ao driver, nunca a base inteira.
    metrics = frame.agg(*expressions).first()
    rows = [
        (source_name, name, metrics[f"null_{index}"], metrics[f"blank_{index}"],
         metrics[f"space_{index}"], metrics[f"distinct_{index}"],
         metrics[f"trim_distinct_{index}"])
        for index, name in enumerate(frame.columns)
    ]
    return spark.createDataFrame(
        rows,
        "fonte string, coluna string, nulos long, vazios long, "
        "com_espacos_nas_bordas long, distintos_originais long, distintos_apos_trim long",
    )


display(column_profile(sales_raw, "sales").unionByName(
    column_profile(channels_raw, "channel_group")
))

fonte,coluna,nulos,vazios,com_espacos_nas_bordas,distintos_originais,distintos_apos_trim
sales,DATE,0,0,0,13,13
sales,CE_BRAND_FLVR,0,0,0,4,4
sales,BRAND_NM,0,0,16151,4,4
sales,Btlr_Org_LVL_C_Desc,0,0,0,7,7
sales,CHNL_GROUP,0,0,0,15,15
sales,TRADE_CHNL_DESC,0,0,0,31,31
sales,PKG_CAT,0,0,0,3,3
sales,Pkg_Cat_Desc,0,0,0,3,3
sales,TSR_PCKG_NM,0,0,0,8,8
sales,$ Volume,0,0,0,2911,2911


## 3 Validade dos valores e coerência temporal

Verifica se datas e valores podem ser interpretados e se ano e mês correspondem à data. Sinaliza valores negativos e zeros para investigação.

In [0]:
sales_typed = (
    sales_raw
    .withColumn("sale_date", F.try_to_timestamp(F.trim(F.col("DATE")), F.lit("M/d/yyyy")).cast("date"))
    .withColumn("sales_amount", F.expr("try_cast(trim(`$ Volume`) AS decimal(28,6))"))
    .withColumn("source_year", F.expr("try_cast(trim(`YEAR`) AS int)"))
    .withColumn("source_month", F.expr("try_cast(trim(`MONTH`) AS int)"))
    .withColumn("source_period", F.expr("try_cast(trim(`PERIOD`) AS int)"))
)

checks = [
    ("data_ausente_ou_nao_convertivel", F.col("sale_date").isNull()),
    ("valor_ausente_ou_nao_convertivel", F.col("sales_amount").isNull()),
    ("formato_valor_fora_decimal_simples",
     ~F.coalesce(F.trim(F.col("$ Volume")).rlike(r"^[+-]?\d+(\.\d+)?$"), F.lit(False))),
    ("valor_com_mais_de_6_casas", F.trim(F.col("$ Volume")).rlike(r"\.\d{7,}$")),
    ("valor_negativo_investigar", F.col("sales_amount") < 0),
    ("valor_zero_investigar", F.col("sales_amount") == 0),
    ("ano_ausente_ou_nao_convertivel", F.col("source_year").isNull()),
    ("mes_ausente_ou_fora_de_1_a_12",
     F.col("source_month").isNull() | ~F.col("source_month").between(1, 12)),
    ("periodo_ausente_ou_nao_convertivel", F.col("source_period").isNull()),
    ("ano_diverge_da_data", F.col("source_year") != F.year("sale_date")),
    ("mes_diverge_da_data", F.col("source_month") != F.month("sale_date")),
]
check_totals = sales_typed.agg(*[
    F.count(F.when(condition, 1)).alias(name) for name, condition in checks
]).first()
display(spark.createDataFrame(
    [(name, check_totals[name]) for name, _ in checks],
    "verificacao string, registros_sinalizados long",
))
for name, condition in checks:
    if check_totals[name]:
        print(f"Amostra — {name}: {check_totals[name]} registros")
        display(sales_typed.filter(condition).limit(20))

verificacao,registros_sinalizados
data_ausente_ou_nao_convertivel,0
valor_ausente_ou_nao_convertivel,0
formato_valor_fora_decimal_simples,0
valor_com_mais_de_6_casas,0
valor_negativo_investigar,260
valor_zero_investigar,127
ano_ausente_ou_nao_convertivel,0
mes_ausente_ou_fora_de_1_a_12,0
periodo_ausente_ou_nao_convertivel,0
ano_diverge_da_data,0


Amostra — valor_negativo_investigar: 260 registros


DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,$ Volume,YEAR,MONTH,PERIOD,sale_date,sales_amount,source_year,source_month,source_period
1/1/2006,3441,RASPBERRY,SOUTHEAST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,2006-01-01,-4.220000,2006,1,1
1/1/2006,3440,LEMON,CANADA,OTHER SMALL STORES,ALL OTHER,N20O,20Z/600ML,.591L NRP 24L,-7.49,2006,1,1,2006-01-01,-7.490000,2006,1,1
1/1/2006,3440,LEMON,SOUTHEAST,LEISURE,AMUSEMENT/RECREATIONAL,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,2006-01-01,-4.220000,2006,1,1
1/1/2006,3554,STRAWBERRY,GREAT LAKES,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P S,-8.45,2006,1,1,2006-01-01,-8.450000,2006,1,1
1/1/2006,3440,LEMON,SOUTHEAST,HEALTHCARE,HEALTH / HOSPITAL,N56P,500ML 6P,.5L NRP 6P,-2.11,2006,1,1,2006-01-01,-2.110000,2006,1,1
1/1/2006,3441,RASPBERRY,CANADA,WHOLESALE,WHOLESALE,N56P,500ML 6P,.5L NRP 6P*,-2.11,2006,1,1,2006-01-01,-2.110000,2006,1,1
1/1/2006,3440,LEMON,SOUTHEAST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,-4.22,2006,1,1,2006-01-01,-4.220000,2006,1,1
1/1/2006,3441,RASPBERRY,GREAT LAKES,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P S,-27.47,2006,1,1,2006-01-01,-27.470000,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHEAST,HEALTHCARE,HEALTH / HOSPITAL,N56P,500ML 6P,.5L NRP 6P,-2.11,2006,1,1,2006-01-01,-2.110000,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHEAST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,-2.11,2006,1,1,2006-01-01,-2.110000,2006,1,1


Amostra — valor_zero_investigar: 127 registros


DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,$ Volume,YEAR,MONTH,PERIOD,sale_date,sales_amount,source_year,source_month,source_period
1/1/2006,3554,STRAWBERRY,NORTHEAST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3440,LEMON,WEST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHWEST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHWEST,CONVENIENCE RETAIL,CONVENIENCE STORE,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3441,RASPBERRY,CANADA,OTHER SMALL STORES,ALL OTHER,N20O,20Z/600ML,.591L NRP 24L *,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3554,STRAWBERRY,GREAT LAKES,OTHER SMALL STORES,ALL OTHER,N20O,20Z/600ML,20z NRP 24L S,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3440,LEMON,SOUTHEAST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3440,LEMON,SOUTHWEST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3554,STRAWBERRY,SOUTHEAST,OTHER SMALL STORES,ALL OTHER,N56P,500ML 6P,.5L NRP 6P,0,2006,1,1,2006-01-01,0.000000,2006,1,1
1/1/2006,3441,RASPBERRY,SOUTHEAST,FOOD SERVICE,BAR/TAVERN/LICENSED EST,N20O,20Z/600ML,20Z NRP 24L,0,2006,1,1,2006-01-01,0.000000,2006,1,1


## 4 Referências de reconciliação

Apresenta o intervalo de datas, os extremos e a soma das vendas. Os totais e a distribuição mensal servem de referência para conferir as próximas etapas; falhas de conversão devem ser consideradas na interpretação.

In [0]:
display(sales_typed.agg(
    F.count("*").alias("registros"),
    F.count("sales_amount").alias("valores_convertidos"),
    F.min("sale_date").alias("primeira_data"),
    F.max("sale_date").alias("ultima_data"),
    F.min("sales_amount").alias("menor_valor"),
    F.max("sales_amount").alias("maior_valor"),
    F.sum("sales_amount").alias("soma_valores_convertidos"),
))
display(
    sales_typed.groupBy(F.date_format("sale_date", "yyyy-MM").alias("ano_mes"))
    .agg(F.count("*").alias("registros"), F.sum("sales_amount").alias("vendas"))
    .orderBy("ano_mes")
)
display(
    sales_raw.groupBy("YEAR", "MONTH", "PERIOD").count().orderBy("YEAR", "MONTH", "PERIOD")
)

registros,valores_convertidos,primeira_data,ultima_data,menor_valor,maior_valor,soma_valores_convertidos
16151,16151,2006-01-01,2006-03-29,-797.500000,8700.000000,4348925.160000


ano_mes,registros,vendas
2006-01,4784,1214299.570000
2006-02,4868,1306120.770000
2006-03,6499,1828504.820000


YEAR,MONTH,PERIOD,count
2006,1,1,1137
2006,1,2,1204
2006,1,3,1218
2006,1,4,1225
2006,2,1,1225
2006,2,2,1214
2006,2,3,1216
2006,2,4,1213
2006,3,1,1292
2006,3,2,1291


## 5 Repetições exatas

Identifica linhas com todos os campos iguais e mede sua multiplicidade. Repetição é um indício a investigar, não uma justificativa automática para exclusão.

In [0]:
for source_name, frame in [("sales", sales_raw), ("channel_group", channels_raw)]:
    repeated = frame.groupBy(*frame.columns).count().filter(F.col("count") > 1)
    print(f"Repetições exatas — {source_name}")
    display(repeated.agg(
        F.count("*").alias("grupos_repetidos"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("linhas_nos_grupos"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("ocorrencias_alem_da_primeira"),
    ))
    display(repeated.orderBy(F.desc("count")).limit(20))

Repetições exatas — sales


grupos_repetidos,linhas_nos_grupos,ocorrencias_alem_da_primeira
0,0,0


DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,$ Volume,YEAR,MONTH,PERIOD,count


Repetições exatas — channel_group


grupos_repetidos,linhas_nos_grupos,ocorrencias_alem_da_primeira
0,0,0


TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC,count


## Leitura dos resultados

Relacionar os problemas encontrados às regras de qualidade necessárias e registrar os totais de referência.

As justificativas técnicas e as propostas de tratamento estão em [README](README.md).